# Phase 1 — Face CNN (FER2013)

**Thesis:** Multimodal Depression Risk Detection  
**Author:** Md. Mursalin, Netrokona University

This notebook trains the **Face Branch CNN** on FER2013 facial expression data.
The trained model will later be used in Phase 4 (multimodal fusion with audio).

---
### What happens here:
1. Clone the repo & install dependencies
2. Download FER2013 from Kaggle
3. Preprocess (augmentation: flip + brightness)
4. Train FaceCNN (3-block Conv → FC → 7-class emotion)
5. Evaluate on test set
6. Save the trained model weights

## 1. Setup

In [ ]:
# Clone the thesis repo
!git clone https://github.com/DevMursLab/Thesis.git depression_thesis
%cd depression_thesis
!pip install -q -r requirements.txt

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))

## 2. Download FER2013 Dataset

In [ ]:
# Option A: Kaggle API (recommended on Colab)
# Upload your kaggle.json first, then run:

from google.colab import files
print('Upload your kaggle.json')
files.upload()

In [ ]:
import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d msambare/fer2013 -p data/raw/ --unzip
!ls data/raw/

## 3. Preprocess FER2013

In [ ]:
!python src/preprocessing/face_preprocess.py

In [ ]:
import numpy as np

X_train = np.load('data/processed/X_train.npy')
y_train = np.load('data/processed/y_train.npy')
X_test  = np.load('data/processed/X_test.npy')
y_test  = np.load('data/processed/y_test.npy')

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Pixel range: [{X_train.min():.2f}, {X_train.max():.2f}]')

## 4. Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt

EMOTIONS = ['Angry','Disgust','Fear','Happy','Sad','Surprise','Neutral']

fig, axes = plt.subplots(2, 7, figsize=(14, 4))
for cls in range(7):
    idx = np.where(y_train == cls)[0][0]
    axes[0, cls].imshow(X_train[idx, 0], cmap='gray')
    axes[0, cls].set_title(EMOTIONS[cls], fontsize=9)
    axes[0, cls].axis('off')
    idx2 = np.where(y_train == cls)[0][1]
    axes[1, cls].imshow(X_train[idx2, 0], cmap='gray')
    axes[1, cls].axis('off')
plt.suptitle('FER2013 — Sample Images per Emotion Class', y=1.02)
plt.tight_layout()
plt.savefig('results/figures/fer2013_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Train Face CNN

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import tqdm

from src.models.face_cnn import FaceCNN
from configs.config import BATCH_SIZE, LEARNING_RATE, EPOCHS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Training on:', device)

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_ds  = TensorDataset(torch.tensor(X_test),  torch.tensor(y_test))
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model     = FaceCNN(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

history = {'train_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    # Train
    model.train()
    total_loss = 0
    for xb, yb in tqdm(train_dl, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    # Eval
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total   += yb.size(0)
    acc = 100 * correct / total

    history['train_loss'].append(total_loss / len(train_dl))
    history['val_acc'].append(acc)
    print(f'Epoch {epoch+1:02d}/{EPOCHS} — loss: {history["train_loss"][-1]:.4f}  val_acc: {acc:.2f}%')

## 6. Results & Plots

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss']); ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch')
ax2.plot(history['val_acc'], color='green'); ax2.set_title('Validation Accuracy (%)'); ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('results/figures/phase1_training_curves.png', dpi=150)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

all_preds, all_true = [], []
model.eval()
with torch.no_grad():
    for xb, yb in test_dl:
        all_preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())

cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=EMOTIONS, yticklabels=EMOTIONS, cmap='Blues')
plt.title('Confusion Matrix — Face CNN (FER2013)')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('results/figures/phase1_confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(all_true, all_preds, target_names=EMOTIONS))

## 7. Save Model Weights

In [ ]:
import os, json

os.makedirs('results/metrics', exist_ok=True)
torch.save(model.state_dict(), 'results/face_cnn_phase1.pth')
print('Model saved: results/face_cnn_phase1.pth')

final_metrics = {
    'phase': 1,
    'dataset': 'FER2013',
    'epochs': EPOCHS,
    'final_val_acc': history['val_acc'][-1],
    'best_val_acc': max(history['val_acc']),
}
with open('results/metrics/phase1_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)
print('Metrics saved:', final_metrics)